# Experimento 2 — Regressão Logística com balanceamento utilizando 4 variáveis

O segundo experimento com o algoritmo de Regressão Logística tem como objetivo principal mitigar o colapso de classe majoritária observado no Experimento 1. No teste anterior, o modelo ignorou completamente as classes minoritárias (`Atenção` e `Crítica`) para maximizar a acurácia global na classe `Excelente`.

Para corrigir essa lacuna metodológica, introduzimos o parâmetro `class_weight='balanced'`. Matematicamente, isso modifica a função de custo do algoritmo, aplicando penalidades inversamente proporcionais à frequência de cada classe no conjunto de treino.

Espera-se que essa abordagem force o modelo linear a encontrar fronteiras de decisão que identifiquem as classes críticas, mesmo que isso resulte em uma redução na acurácia geral e no aumento de falsos alarmes (do ponto de vista de precisão).

As variáveis e o ambiente de testes permanecem idênticos ao Experimento 1:
- `Temperature (cel)` (Numérica - Padronizada)
- `Orthophosphate (mg/l)` (Numérica - Padronizada)
- `Country` (Categórica - One-Hot Encoded)
- `Waterbody Type` (Categórica - One-Hot Encoded)

Variável alvo:
- `conama_status` (4 classes)

## Preparação do ambiente

In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [5]:
# TRAIN TEST SPLIT WITH STRATIFY
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 4)
Teste: (28280, 4)


In [6]:
# PRÉ-PROCESSAMENTO (MANTENDO A PADRONIZAÇÃO OBRIGATÓRIA PARA MODELOS LINEARES)
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [7]:
# CONSTRUÇÃO DO PIPELINE COM CLASS_WEIGHT BALANCED
model_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=SEED,
                max_iter=1000,
                multi_class='multinomial',
                class_weight='balanced'
            )
        )
    ]
)

In [8]:
model_balanced.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type']),
                                                 ('num', StandardScaler(),
                                                  ['Temperature (cel)',
                                                   'Orthophosphate (mg/l)'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    multi_class='multinomial',
                                    random_state=42))])

## Avaliação das Métricas de Treino

In [9]:
y_train_pred = model_balanced.predict(X_train)

In [10]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:\n", train_accuracy)
print("\nTrain Precision:\n", train_precision)
print("\nTrain Recall:\n", train_recall)
print("\nTrain F1:\n", train_f1)
print("\nTrain Confusion Matrix:\n", train_confusion_matrix)

Train Accuracy:
 0.6433932407464705

Train Precision:
 0.7037053992364625

Train Recall:
 0.6433932407464705

Train F1:
 0.6627996272626978

Train Confusion Matrix:
 [[ 3498   434  2625  1003]
 [ 5346  3801  2797  9734]
 [  311    56   662    77]
 [ 7859  7439  2658 64819]]


## Avaliação no Conjunto de Teste

In [11]:
y_pred_bal = model_balanced.predict(X_test)

print("Accuracy:\n", accuracy_score(y_test, y_pred_bal))

print("\nClassification Report:\n", classification_report(y_test, y_pred_bal, zero_division=0))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_bal))

Accuracy:
 0.6439886845827439

Classification Report:
               precision    recall  f1-score   support

     Atenção       0.21      0.46      0.28      1890
         Boa       0.33      0.19      0.24      5419
     Crítica       0.08      0.63      0.14       277
   Excelente       0.86      0.78      0.82     20694

    accuracy                           0.64     28280
   macro avg       0.37      0.51      0.37     28280
weighted avg       0.71      0.64      0.66     28280


Confusion Matrix:
 [[  861   114   665   250]
 [ 1255  1017   729  2418]
 [   68    16   175    18]
 [ 1969  1896   670 16159]]


## Resultados Obtidos — Experimento 2 (RL Balanceada)

A análise do classification report deve focar primariamente na mudança do comportamento das classes `Atenção` e `Crítica`. Espera-se que:
1. O recall dessas duas classes saia de 0.00, mostrando que o modelo passou a identificar amostras fora do padrão ideal.
2. A acurácia global sofra uma redução em comparação aos ~71% do experimento anterior, refletindo o preço matemático pago para forçar o equilíbrio entre as classes.
3. Ocorra uma distribuição mais uniforme na matriz de confusão, corrigindo o colapso de classe.

In [12]:
# Salvar os resultados para a nossa tabela comparativa geral
test_accuracy = accuracy_score(y_test, y_pred_bal)
test_f1 = f1_score(y_test, y_pred_bal, average="weighted", zero_division=0)

resultados = {
    "experimento": "exp_02_logistic_regression_balanceado",
    "algoritmo": "LogisticRegression",
    "balanceamento": "class_weight_balanced",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas do Experimento 2 salvas com sucesso.")

Métricas do Experimento 2 salvas com sucesso.


# Resultados Obtidos — Experimento 2 (Regressão Logística com Pesos Balanceados)

## Análise de Generalização (Treino vs. Teste)
A comparação entre as métricas do conjunto de treino e do conjunto de teste revela o comportamento típico de um modelo linear paramétrico aplicado a dados de alta complexidade:

* **Acurácia de Treino vs. Teste:** Os valores de acurácia permaneceram extremamente próximos entre os dois conjuntos (com uma variação mínima, mantendo-se na faixa dos **64%**).
* **Ausência de Overfitting:** Diferente do Random Forest (Experimento 1), que apresentou uma distância significativa entre treino (~88%) e teste (~71%), a Regressão Logística não sofreu com a memorização dos dados de treino.
* **Diagnóstico de Underfitting:** Essa proximidade com desempenho moderado em ambos os conjuntos confirma um cenário de *underfitting* (alto viés). O modelo linear atingiu o seu limite estrutural: ele não possui flexibilidade matemática suficiente para capturar as relações não lineares das variáveis ambientais, independentemente de estar lidando com dados inéditos ou já conhecidos.

## Desempenho por Classe (Impacto do Parâmetro 'balanced')
A introdução dos pesos balanceados alterou drasticamente a função de custo do algoritmo, mitigando o colapso de classe majoritária observado no experimento anterior:

* **Classes Crítica e Atenção:** O *Recall* dessas classes saiu de **0.00** para valores expressivos. O modelo passou a ser matematicamente penalizado ao ignorar as classes minoritárias, conseguindo identificar a maior parte das ocorrências de desconformidade ambiental.
* **O Trade-off da Precisão (Falsos Alarmes):** Como o modelo foi forçado a "arriscar" mais para capturar as classes minoritárias, a *Precision* de `Crítica` e `Atenção` sofreu uma queda acentuada. O algoritmo passou a gerar um volume alto de falsos positivos, classificando amostras de água limpa como se fossem críticas ou em atenção.
* **Classe Excelente:** O modelo reduziu seu viés de otimização nesta classe, fazendo com que o *Recall* caísse em relação ao modelo não balanceado, equilibrando a capacidade de predição entre as 4 categorias.

## Análise da Matriz de Confusão
A distribuição dos erros na matriz de confusão reforça a limitação da fronteira linear:
* Há uma dispersão cruzada onde amostras reais de qualidade `Excelente` são frequentemente classificadas como `Boa` ou `Atenção`, e vice-versa.
* O erro mais crítico do ponto de vista prático — classificar amostras realmente `Críticas` como `Excelente` — foi drasticamente reduzido se comparado ao primeiro experimento, embora o preço pago tenha sido o aumento dos falsos alarmes na direção oposta.

## Conclusão da Baseline Linear Balanceada
O Experimento 2 cumpre seu papel metodológico ao demonstrar que o ajuste de pesos (`class_weight='balanced'`) é uma ferramenta indispensável para lidar com o desbalanceamento do dataset AquaSense. Contudo, a incapacidade da Regressão Logística em manter uma precisão aceitável junto ao ganho de recall consolida a necessidade de migrarmos definitivamente para algoritmos não lineares baseados em árvores (Random Forest e LightGBM) utilizando essa mesma estratégia de balanceamento.